# LMC2 #333, adım 3: post-processing'in validation split'te kontrolü

Held-out çalışmalarda seçilen ayarlar (Nucleus, Tubulin, Actin için TV 0.1, Mitochondria için gamma 1.25) validation split'e uygulanır. Yalnızca bu iki ayar ve değişmemiş tahmin skorlanır. L4 ile Tümünü çalıştır.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, json, glob
R = '/content/lmc2-task1-w1'
if not os.path.exists(R):
    !git clone -q https://github.com/neuronautsai/lmc2-task1-w1 {R}
!git -C {R} pull -q && git -C {R} log -1 --oneline
!pip -q install segmentation-models-pytorch==0.6.0 xmltodict
%cd {R}
!nvidia-smi --query-gpu=name,memory.total --format=csv

D = '/content/drive/MyDrive/LMC2'
W5 = f'{D}/outputs/w5'
PAIRS = f'{D}/outputs/dup_check/pairs_excluded.csv'
SMP = [f'{W5}/smp_r34_s42_v2/best.pt', f'{W5}/smp_r34_s123/best.pt', f'{W5}/smp_r34_s456/best.pt']
BASE_HO = f'{W5}/base_heldout_s42/best.pt'
BASE3 = [f'{D}/outputs/w3/base_s{s}/best.pt' for s in (42, 123, 456)]
HELDOUT = f'{W5}/smp_r34_s42_v2/heldout_studies.json'
SIZE = 3000
for p in [PAIRS, *SMP, BASE_HO, *BASE3, HELDOUT]:
    print(os.path.exists(p), p)
    assert os.path.exists(p), f'eksik: {p}'
S_SMP, S_BASE3, S_ALL = ' '.join(SMP), ' '.join(BASE3), ' '.join(SMP + [BASE_HO] + BASE3)

In [ ]:
O6 = f'{W5}/postprocess_smp3_tta_val_check'
!python postprocess.py --checkpoint {S_SMP} --tta --split val --tv 0.1 --offset --gamma 1.25 --out {O6} --pairs-csv {PAIRS} --resume
!cat {O6}/summary_postprocess.csv